# 03 - Player Features: Multi-Accounting, Payment, and Betting Groups

Purpose: build and verify the reviewed Phase 3 `ma_`, `pay_`, and `bet_` feature groups from the frozen snapshot.

Input: `frauddet.snapshot.load_snapshot` using `config.yaml` `phase3.input_dir`. No live Mongo and no mutable top-level parquet inputs.

Outputs:
- `data/player_features.parquet`: one row per canonical player, wide scalar + metadata columns.
- `data/player_features_evidence.parquet`: one row per `player_key + feature_name`; `feature_evidence` is JSON reviewer evidence.

Evidence uses direct one-hop links only. Identity hashes remain hashed; no raw NIN/email is present. Tier 4 IP/bank/passport and casino game-exploitation features are deferred.


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from frauddet import config
from frauddet.features import build_phase3_features
from frauddet.features.betting import (
    FEATURE_SPECS as BET_FEATURE_SPECS,
    sportsbook_active_players,
)
from frauddet.features.multi_accounting import (
    FEATURE_SPECS as MA_FEATURE_SPECS,
    build_multi_account_linkages,
)
from frauddet.features.payment import FEATURE_SPECS as PAY_FEATURE_SPECS
from frauddet.features.withdrawals import build_withdrawal_context
from frauddet.snapshot import load_snapshot, snapshot_dir

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)
print("snapshot_dir", snapshot_dir())

snapshot_dir C:\Users\dev\OneDrive\Documents\Fraud Detection - FS Book\Fraud-Detection-FS-Book\data\snapshot_phase3_2026-06-19b


## Build

In [2]:
result = build_phase3_features()
player_features = result.player_features
feature_evidence = result.feature_evidence
result.output_paths, player_features.shape, feature_evidence.shape

({'player_features': WindowsPath('C:/Users/dev/OneDrive/Documents/Fraud Detection - FS Book/Fraud-Detection-FS-Book/data/player_features.parquet'),
  'feature_evidence': WindowsPath('C:/Users/dev/OneDrive/Documents/Fraud Detection - FS Book/Fraud-Detection-FS-Book/data/player_features_evidence.parquet')},
 (116, 133),
 (3828, 3))

## Feature Coverage

In [3]:
coverage_rows = []
for feature_name, spec in MA_FEATURE_SPECS.items():
    values = player_features[feature_name]
    coverage_rows.append(
        {
            "feature_name": feature_name,
            "strength": spec.strength,
            "scoring_role": spec.scoring_role,
            "players": len(values),
            "non_null": int(values.notna().sum()),
            "non_zero": (
                int(values.dropna().astype(bool).sum())
                if spec.value_kind == "bool"
                else int(values.dropna().ne(0).sum())
            ),
            "null": int(values.isna().sum()),
        }
    )
coverage = pd.DataFrame(coverage_rows)
display(coverage)

assert int(coverage.loc[coverage.feature_name.eq("ma_email_shared_account_count"), "non_zero"].iloc[0]) == 2
assert int(coverage.loc[coverage.feature_name.eq("ma_nin_shared_account_count"), "non_zero"].iloc[0]) == 0
assert int(coverage.loc[coverage.feature_name.eq("ma_identity_phone_collision_count"), "non_zero"].iloc[0]) == 0
assert player_features["ma_nin_shared_account_count"].notna().all()
assert player_features["ma_identity_phone_collision_count"].notna().all()

# Device-link contamination guard: measure all valid fingerprints, but link only
# fingerprints whose population cardinality is at or below the configured cap.
players_raw = load_snapshot("players")
logins_raw = load_snapshot("logins")
money_raw = load_snapshot("money")
linkages = build_multi_account_linkages(players_raw, logins_raw, money_raw)
device_cap = int(config.load_config()["thresholds"]["ma_device_max_cardinality"])
excluded_device_cardinalities = sorted(
    [
        len(player_keys)
        for player_keys in linkages.device_all.key_to_players.values()
        if len(player_keys) > device_cap
    ],
    reverse=True,
)
print("device_max_cardinality", device_cap)
print("excluded_fingerprint_count", len(excluded_device_cardinalities))
print("excluded_fingerprint_cardinalities", excluded_device_cardinalities)
assert all(
    len(player_keys) <= device_cap
    for player_keys in linkages.device.key_to_players.values()
)

,feature_name,strength,scoring_role,players,non_null,non_zero,null
0,ma_nin_shared_account_count,strong,scoring,116,116,0,0
1,ma_email_shared_account_count,strong,scoring,116,116,2,0
2,ma_device_shared_account_count,strong,scoring,116,87,43,29
3,ma_withdrawal_recipient_shared_count,strong,scoring,116,10,0,106
4,ma_identity_phone_collision_count,strong,scoring,116,116,0,0
5,ma_referred_by_linked_account,strong,supporting,116,116,0,0
6,ma_referral_fanout_count,weak,supporting,116,116,2,0
7,ma_cocreated_linked_count,moderate,supporting,116,116,9,0
8,ma_device_count,weak,context_only,116,87,87,29


device_max_cardinality 10
excluded_fingerprint_count 5
excluded_fingerprint_cardinalities [28, 18, 15, 11, 11]


## Evidence Integrity

In [4]:
evidence_lookup = {
    (row.player_key, row.feature_name): json.loads(row.feature_evidence)
    for row in feature_evidence.itertuples(index=False)
}
for row in player_features.itertuples(index=False):
    values = row._asdict()
    for feature_name, spec in MA_FEATURE_SPECS.items():
        value = values[feature_name]
        if spec.scoring_role == "scoring" and pd.notna(value) and bool(value):
            assert evidence_lookup[(values["player_key"], feature_name)]
print("VERIFIED: every non-zero scoring ma_ feature has non-empty evidence")

excluded_fingerprints = {
    shared_key
    for shared_key, player_keys in linkages.device_all.key_to_players.items()
    if len(player_keys) > device_cap
}
for row in feature_evidence.loc[
    feature_evidence.feature_name.isin(
        {
            "ma_device_shared_account_count",
            "ma_referred_by_linked_account",
            "ma_cocreated_linked_count",
        }
    )
].itertuples(index=False):
    assert not any(
        fingerprint in row.feature_evidence
        for fingerprint in excluded_fingerprints
    )
print("VERIFIED: over-cardinality fingerprints are absent from link/corroboration evidence")

VERIFIED: every non-zero scoring ma_ feature has non-empty evidence
VERIFIED: over-cardinality fingerprints are absent from link/corroboration evidence


## Hand Reconciliation Against Frozen Parquet

In [5]:
players_raw = load_snapshot("players")
logins_raw = load_snapshot("logins")
money_raw = load_snapshot("money")

email_players = player_features.loc[
    player_features["ma_email_shared_account_count"].gt(0), "player_key"
].astype(str).tolist()
referral_players = player_features.loc[
    player_features["ma_referred_by_linked_account"].fillna(False), "player_key"
].astype(str).tolist()
device_players = player_features.loc[
    player_features["ma_device_shared_account_count"].fillna(0).gt(0), "player_key"
].astype(str).tolist()
picked_players = list(
    dict.fromkeys(email_players + referral_players[:1] + device_players[:1])
)[:3]

print("picked_players", picked_players)
display(
    player_features.loc[
        player_features.player_key.isin(picked_players),
        ["player_key"] + list(MA_FEATURE_SPECS),
    ]
)

focus_features = {
    "ma_email_shared_account_count",
    "ma_referred_by_linked_account",
    "ma_device_shared_account_count",
}
focus_evidence = feature_evidence[
    feature_evidence.player_key.isin(picked_players)
    & feature_evidence.feature_name.isin(focus_features)
].copy()
focus_evidence["parsed_evidence"] = focus_evidence.feature_evidence.map(json.loads)
display(focus_evidence[["player_key", "feature_name", "parsed_evidence"]])

linked_players = set(picked_players)
for evidence in focus_evidence.feature_evidence.map(json.loads):
    for item in evidence:
        linked_players.update(item.get("other_player_keys", []))
        if item.get("referrer_player_key"):
            linked_players.add(item["referrer_player_key"])

# Raw frozen rows needed to reconcile the selected hashes/referrals/devices.
display(
    players_raw.loc[
        players_raw.player_key.isin(linked_players),
        ["player_key", "created_at", "nin_hash", "email_hash", "referred_by_key"],
    ].sort_values("player_key")
)

shared_fingerprints = set()
for evidence in focus_evidence.feature_evidence.map(json.loads):
    for item in evidence:
        if item.get("shared_key_type") == "fingerprint":
            shared_fingerprints.add(item["shared_key"])
        for linkage in item.get("corroborating_linkages", []):
            if linkage.get("shared_key_type") == "fingerprint":
                shared_fingerprints.add(linkage["shared_key"])

display(
    logins_raw.loc[
        logins_raw.player_key.isin(linked_players)
        & logins_raw.fingerprint.isin(shared_fingerprints),
        ["player_key", "source_id", "fingerprint", "created_at"],
    ].drop_duplicates().sort_values(["fingerprint", "player_key", "created_at"])
)

# No shared-recipient feature fires in this snapshot, but show picked players' completed withdrawals if any.
display(
    money_raw.loc[
        money_raw.player_key.isin(picked_players)
        & money_raw.txn_type.eq("WITHDRAWAL")
        & money_raw.is_money_out.fillna(False),
        ["player_key", "transaction_id", "recipient_normalized", "requested_at"],
    ].sort_values(["player_key", "requested_at"])
)

picked_players ['6a26959dfc099dd782863ecf', '6a311326aa7ed994dfda83af', '6a0ea9ff174ad3c431d9e16d']


,player_key,ma_nin_shared_account_count,ma_email_shared_account_count,ma_device_shared_account_count,ma_withdrawal_recipient_shared_count,ma_identity_phone_collision_count,ma_referred_by_linked_account,ma_referral_fanout_count,ma_cocreated_linked_count,ma_device_count
0,6a0ea9ff174ad3c431d9e16d,0,0,9,0,0,False,0,1,8
41,6a26959dfc099dd782863ecf,0,1,0,<NA>,0,False,0,0,2
84,6a311326aa7ed994dfda83af,0,1,2,0,0,False,0,0,2


,player_key,feature_name,parsed_evidence
12,6a0ea9ff174ad3c431d9e16d,ma_device_shared_account_count,"[{'linked_source_record_ids': {'6a0eabae174ad3c431d9e6b6': ['6a0f1dc0a5d7ce78c657bf45'], '6a0f0df7a5d7ce78c657a62f': ['6a0f0e07a5d7ce78c657a649', '6a0f1e89a..."
13,6a0ea9ff174ad3c431d9e16d,ma_email_shared_account_count,[]
17,6a0ea9ff174ad3c431d9e16d,ma_referred_by_linked_account,[]
1365,6a26959dfc099dd782863ecf,ma_device_shared_account_count,[]
1366,6a26959dfc099dd782863ecf,ma_email_shared_account_count,"[{'linked_source_record_ids': {'6a311326aa7ed994dfda83af': []}, 'other_player_keys': ['6a311326aa7ed994dfda83af'], 'shared_key': '8e922392304ca6feb6fc074475..."
1370,6a26959dfc099dd782863ecf,ma_referred_by_linked_account,[]
2784,6a311326aa7ed994dfda83af,ma_device_shared_account_count,"[{'linked_source_record_ids': {'6a0eac61174ad3c431d9e947': ['6a1041b3a5d7ce78c65809f5', '6a106245a5d7ce78c65819a8', '6a11d71c0a5fadca63729865', '6a140b850a5..."
2785,6a311326aa7ed994dfda83af,ma_email_shared_account_count,"[{'linked_source_record_ids': {'6a26959dfc099dd782863ecf': []}, 'other_player_keys': ['6a26959dfc099dd782863ecf'], 'shared_key': '8e922392304ca6feb6fc074475..."
2789,6a311326aa7ed994dfda83af,ma_referred_by_linked_account,[]


,player_key,created_at,nin_hash,email_hash,referred_by_key
0,6a0ea9ff174ad3c431d9e16d,2026-05-21 06:45:19.652000+00:00,475cd04fd275c003f8b8a2393132d6076582ef46f17dff8ac4ad0ab5b3790965,c2f8254bfd7b70b9729b963ae7883886273a5c739bc682dd079f5543e9ca24a1,NaN
1,6a0eabae174ad3c431d9e6b6,2026-05-21 06:52:30.870000+00:00,NaN,1a29e014b69b8f8e8d31aae3b6b7544b7fd7f0150ce2931e9ab1363c1bd77d9e,NaN
2,6a0eac61174ad3c431d9e947,2026-05-21 06:55:29.137000+00:00,NaN,9ee553e33a1add77b8a3b84e62e8a4fbfb5227c16a7dc739fb3b43a667b83dcf,NaN
6,6a0eaefe9d615bad27c7f786,2026-05-21 07:06:39.115000+00:00,NaN,799cac9a1358fc863496a205c64a5ccda1a869fd7ace70ffc299cbe7662d7e70,NaN
7,6a0ef9cca5d7ce78c65765d9,2026-05-21 12:25:48.725000+00:00,NaN,ee5fa051aabbad3d0458007a4be7246232cdff2cb260811e0038381518c7ffdb,NaN
8,6a0f0df7a5d7ce78c657a62f,2026-05-21 13:51:51.182000+00:00,NaN,3f6d4dccfe89d1ac131b4a3f3d34cb5fd693ba89050e73111567c1993927d598,6a0ef9cca5d7ce78c65765d9
28,6a2161017364bea0f4f5aede,2026-06-04 11:26:57.619000+00:00,1e051f637a49a1680ec9d238412700c294d6488a376b7fd4c1923d4dd432de9b,NaN,NaN
29,6a21621d7364bea0f4f5b21f,2026-06-04 11:31:41.764000+00:00,474187d39d09a2f4a2c4a130500321c06fb813b7161a7fc13c168cbbbc10a3ac,NaN,NaN
30,6a21639b7364bea0f4f5b356,2026-06-04 11:38:03.863000+00:00,7fd091345fa7dd2973f9e8fc55b5dc89f048f5352ef94fc48b2a90fa299c5159,NaN,NaN
37,6a22bb07ceed05a02d03c8c7,2026-06-05 12:03:19.043000+00:00,dea4f18ac53d2036507a8f5d72eac12f8c284f2327f18f880920aa4a917b3b3a,NaN,NaN


,player_key,source_id,fingerprint,created_at
60,6a0ea9ff174ad3c431d9e16d,6a0f0241a5d7ce78c65781b8,1a42e272f81931ac002de1196d54742e0fce3a1cff42b79a9a5aad42047e0167,2026-05-21 13:01:53.275000+00:00
114,6a0ea9ff174ad3c431d9e16d,6a104c4da5d7ce78c6580f7f,1a42e272f81931ac002de1196d54742e0fce3a1cff42b79a9a5aad42047e0167,2026-05-22 12:30:05.340000+00:00
137,6a0ea9ff174ad3c431d9e16d,6a11fbfb0a5fadca6372aab7,1a42e272f81931ac002de1196d54742e0fce3a1cff42b79a9a5aad42047e0167,2026-05-23 19:11:55.370000+00:00
144,6a0ea9ff174ad3c431d9e16d,6a13359f0a5fadca6372bbd8,1a42e272f81931ac002de1196d54742e0fce3a1cff42b79a9a5aad42047e0167,2026-05-24 17:30:07.574000+00:00
70,6a0eabae174ad3c431d9e6b6,6a0f1dc0a5d7ce78c657bf45,1a42e272f81931ac002de1196d54742e0fce3a1cff42b79a9a5aad42047e0167,2026-05-21 14:59:12.087000+00:00
...,...,...,...,...
284,6a2161017364bea0f4f5aede,6a1c265ca749eff07ed9315e,d922849ec0d12e5598ddcc188d24b1e4b6ed6be07c0817fb50ec5affbf76eb68,2026-05-31 12:15:24.281000+00:00
277,6a22bb07ceed05a02d03c8c7,6a1c0874a749eff07ed924ce,d922849ec0d12e5598ddcc188d24b1e4b6ed6be07c0817fb50ec5affbf76eb68,2026-05-31 10:07:48.878000+00:00
287,6a22bb07ceed05a02d03c8c7,6a1c27f8a749eff07ed932bf,d922849ec0d12e5598ddcc188d24b1e4b6ed6be07c0817fb50ec5affbf76eb68,2026-05-31 12:22:16.829000+00:00
288,6a22bb07ceed05a02d03c8c7,6a1c2849a749eff07ed9331e,d922849ec0d12e5598ddcc188d24b1e4b6ed6be07c0817fb50ec5affbf76eb68,2026-05-31 12:23:37.243000+00:00


,player_key,transaction_id,recipient_normalized,requested_at
43,6a0ea9ff174ad3c431d9e16d,TXNY_MPWO99LH_81C8EA28AE08,757575757,2026-06-02 13:27:09.384000+00:00
44,6a0ea9ff174ad3c431d9e16d,TXNY_MPWOA1IW_1CE1BC90EF45,757575757,2026-06-02 13:27:45.549000+00:00
1138,6a311326aa7ed994dfda83af,TXNY_MQGIA50Q_275478271E03,755432100,2026-06-16 10:35:15.894000+00:00


## FINDINGS — Multi-accounting features — frozen Phase 3 snapshot — 2026-06-22
- Verified: the combined `player_features.parquet` has one row for each of 116 canonical snapshot players and retains all nine requested Tier 1-3 `ma_` features alongside the reviewed `pay_` group.
- Verified: evidence store schema remains long-form `(player_key, feature_name, feature_evidence)` with JSON evidence. The `ma_` group contributes 1,044 rows (116 players x 9 features); non-zero scoring features always have evidence, while zero/null values may honestly carry empty evidence.
- Verified: device sharing uses `ma_device_max_cardinality=10`. Five office/public fingerprints were excluded with player cardinalities `[28, 18, 15, 11, 11]`; excluded hashes are absent from device-link, referral-corroboration, and co-creation evidence.
- Verified: `ma_nin_shared_account_count` is non-null for 116/116 and non-zero for 0; absent NIN correctly maps to measured `0`, not null.
- Verified: `ma_email_shared_account_count` is non-null for 116/116 and non-zero for 2, representing the single frozen shared-email pair.
- Verified: `ma_device_shared_account_count` is measurable for 87/116 and non-zero for 43 after the cap (previously 85); 29 players remain null with `no_valid_fingerprint_logins`. The residual 43/87 is still dominated by sub-cap dev artifacts (test rigs/shared development machines), not genuine device-sharing signal; the uncalibrated PLACEHOLDER cap of 10 requires production calibration and the residual must not be read as signal.
- Verified: `ma_withdrawal_recipient_shared_count` is measurable for 10/116 and non-zero for 0; 106 players are null with `no_completed_withdrawals`.
- Verified: `ma_identity_phone_collision_count` is non-null for 116/116 and non-zero for 0, confirming the dormant phone-collision path is wired.
- Verified: `ma_referred_by_linked_account` is non-null for 116/116 and non-zero for 0 after the cap (previously 3); the previous hits were entirely office-fingerprint contamination. It is now fully dormant on dev, joining NIN and phone collision as built but unexercised paths.
- Verified: `ma_referral_fanout_count` is non-null for 116/116 and non-zero for 2.
- Verified: `ma_cocreated_linked_count` is non-null for 116/116 and non-zero for 9 after the cap (previously 29), using the configured 15-minute window plus a direct non-excluded shared signal.
- Verified: `ma_device_count` remains measurable/non-zero for 87/116 because it counts all valid observed fingerprints, including public/office devices for context; 29 are null with `no_valid_fingerprint_logins`.
- Surprises: Capping five high-cardinality fingerprints removed every referral-linked hit and two-thirds of co-created-linked hits, confirming the uncapped office fingerprints were contaminating stronger features.
- Gaps: Tier 4 IP/bank/passport features remain documented TODOs only. The device cap is a placeholder and must be recalibrated on production data.
- Decisions needed: None for the reviewed `ma_` group. Payment verification follows below.
- Updated assumptions: high-cardinality fingerprints remain visible to `ma_device_count` as context but cannot create device links or corroborate referral/co-creation features.

## Payment Feature Coverage and Evidence Integrity

In [6]:
payment_coverage_rows = []
for feature_name, spec in PAY_FEATURE_SPECS.items():
    values = player_features[feature_name]
    non_zero = (
        int(values.dropna().astype(bool).sum())
        if spec.value_kind == "bool"
        else int(values.dropna().ne(0).sum())
    )
    payment_coverage_rows.append(
        {
            "feature_name": feature_name,
            "strength": spec.strength,
            "scoring_role": spec.scoring_role,
            "players": len(values),
            "non_null": int(values.notna().sum()),
            "non_zero": non_zero,
            "null": int(values.isna().sum()),
        }
    )
payment_coverage = pd.DataFrame(payment_coverage_rows)
display(payment_coverage)

casino_null_count = int(
    player_features["pay_deposit_then_exit_flag__null_reason"]
    .eq("casino_activity_not_observable")
    .sum()
)
print("casino_activity_not_observable players", casino_null_count)
assert casino_null_count == 96

for row in player_features.itertuples(index=False):
    values = row._asdict()
    for feature_name, spec in PAY_FEATURE_SPECS.items():
        value = values[feature_name]
        if spec.scoring_role == "scoring" and pd.notna(value) and bool(value):
            assert evidence_lookup[(values["player_key"], feature_name)]
print("VERIFIED: every non-zero scoring pay_ feature has non-empty evidence")

bets_raw = load_snapshot("bets")
withdrawal_context = build_withdrawal_context(players_raw, money_raw)
shared_linkages = build_multi_account_linkages(
    players_raw, logins_raw, money_raw, withdrawal_context=withdrawal_context
)
assert shared_linkages.recipient is withdrawal_context.recipient
print("VERIFIED: ma_ and pay_ use the same completed-withdrawal recipient context")

,feature_name,strength,scoring_role,players,non_null,non_zero,null
0,pay_deposit_then_exit_flag,strong,scoring,116,8,1,108
1,pay_intervening_turnover_ratio,strong,scoring,116,1,0,115
2,pay_min_minutes_deposit_to_withdrawal,strong,scoring,116,10,10,106
3,pay_third_party_withdrawal_flag,strong,scoring,116,10,0,106
4,pay_third_party_withdrawal_count,strong,scoring,116,10,0,106
5,pay_withdrawal_to_deposit_ratio,moderate,supporting,116,26,9,90
6,pay_fast_withdrawal_count,moderate,supporting,116,10,6,106
7,pay_manual_reconciliation_count,moderate,supporting,116,38,5,78
8,pay_manual_reconciliation_ratio,moderate,supporting,116,38,5,78
9,pay_declined_withdrawal_count,moderate,supporting,116,11,3,105


casino_activity_not_observable players 96
VERIFIED: every non-zero scoring pay_ feature has non-empty evidence
VERIFIED: ma_ and pay_ use the same completed-withdrawal recipient context


## Payment Hand Reconciliation Against Frozen Parquet

In [7]:
flag_players = player_features.loc[
    player_features["pay_deposit_then_exit_flag"].fillna(False), "player_key"
].astype(str).tolist()
fast_players = player_features.loc[
    player_features["pay_fast_withdrawal_count"].fillna(0).gt(0), "player_key"
].astype(str).tolist()
manual_players = player_features.loc[
    player_features["pay_manual_reconciliation_count"].fillna(0).gt(0), "player_key"
].astype(str).tolist()
pay_picked_players = list(
    dict.fromkeys(flag_players + fast_players + manual_players)
)[:3]
print("pay_picked_players", pay_picked_players)
display(
    player_features.loc[
        player_features.player_key.isin(pay_picked_players),
        ["player_key"] + list(PAY_FEATURE_SPECS),
    ]
)

pay_focus_evidence = feature_evidence[
    feature_evidence.player_key.isin(pay_picked_players)
    & feature_evidence.feature_name.str.startswith("pay_")
].copy()
pay_focus_evidence["parsed_evidence"] = pay_focus_evidence.feature_evidence.map(json.loads)
display(
    pay_focus_evidence.loc[
        pay_focus_evidence.parsed_evidence.map(bool),
        ["player_key", "feature_name", "parsed_evidence"],
    ]
)

display(
    money_raw.loc[
        money_raw.player_key.isin(pay_picked_players),
        [
            "player_key", "txn_type", "transaction_id", "amount",
            "final_status", "requested_at", "finalized_at",
            "recipient_normalized", "is_third_party_recipient",
            "is_money_in", "is_money_out",
        ],
    ].sort_values(["player_key", "requested_at", "transaction_id"])
)
display(
    bets_raw.loc[
        bets_raw.player_key.isin(pay_picked_players),
        ["player_key", "ticket_id", "stake", "created_at", "status"],
    ].sort_values(["player_key", "created_at", "ticket_id"])
)

pay_picked_players ['6a2f98f60731c9a7d6634114', '6a0eac6b174ad3c431d9e988', '6a21639b7364bea0f4f5b356']


,player_key,pay_deposit_then_exit_flag,pay_intervening_turnover_ratio,pay_min_minutes_deposit_to_withdrawal,pay_third_party_withdrawal_flag,pay_third_party_withdrawal_count,pay_withdrawal_to_deposit_ratio,pay_fast_withdrawal_count,pay_manual_reconciliation_count,pay_manual_reconciliation_ratio,pay_declined_withdrawal_count,pay_failed_withdrawal_count,pay_net_money_flow,pay_distinct_payment_methods,pay_distinct_payment_accounts
3,6a0eac6b174ad3c431d9e988,False,<NA>,1.2181,False,0,0.1,1,0,0.0,0,0,18000.0,1,1
30,6a21639b7364bea0f4f5b356,False,<NA>,0.832933,False,0,0.053712,4,0,0.0,3,1,2452481.0,2,1
81,6a2f98f60731c9a7d6634114,True,0.0,4.494317,False,0,0.136364,2,0,0.0,0,0,20900.0,1,1


,player_key,feature_name,parsed_evidence
120,6a0eac6b174ad3c431d9e988,pay_distinct_payment_accounts,[{'payment_accounts': ['0777777777']}]
121,6a0eac6b174ad3c431d9e988,pay_distinct_payment_methods,[{'payment_methods': ['mobile_money']}]
123,6a0eac6b174ad3c431d9e988,pay_fast_withdrawal_count,"[{'deposit_id': 'TXNY_MQ6HT7DX_79F428E3BA18', 'withdrawal_id': 'TXNY_MQ6HUWZA_123847B72D92'}]"
127,6a0eac6b174ad3c431d9e988,pay_min_minutes_deposit_to_withdrawal,"[{'deposit_id': 'TXNY_MQ6HT7DX_79F428E3BA18', 'gap_minutes': 1.2181, 'withdrawal_id': 'TXNY_MQ6HUWZA_123847B72D92'}]"
128,6a0eac6b174ad3c431d9e988,pay_net_money_flow,"[{'money_in_sum_ugx': 20000.0, 'money_out_sum_ugx': 2000.0}]"
131,6a0eac6b174ad3c431d9e988,pay_withdrawal_to_deposit_ratio,"[{'money_in_sum_ugx': 20000.0, 'money_out_sum_ugx': 2000.0, 'reason_text': 'completed money_out UGX 2000 / completed money_in UGX 20000'}]"
1009,6a21639b7364bea0f4f5b356,pay_declined_withdrawal_count,"[{'withdrawal_ids': ['TXNY_MQ0WUD50_EECB042EEF1D', 'TXNY_MQ6MZYGP_2BA1CA017B7B', 'TXNY_MQJ59BQS_44B67A1C0A56']}]"
1011,6a21639b7364bea0f4f5b356,pay_distinct_payment_accounts,[{'payment_accounts': ['0759879879']}]
1012,6a21639b7364bea0f4f5b356,pay_distinct_payment_methods,"[{'payment_methods': ['Wallet', 'mobile_money']}]"
1013,6a21639b7364bea0f4f5b356,pay_failed_withdrawal_count,[{'withdrawal_ids': ['TXNY_MQ6P93B4_E6550489DD8A']}]


,player_key,txn_type,transaction_id,amount,final_status,requested_at,finalized_at,recipient_normalized,is_third_party_recipient,is_money_in,is_money_out
5,6a0eac6b174ad3c431d9e988,DEPOSIT,TXNY_MPNZB81Y_F8B90B86232B,10000,initiated,2026-05-27 11:26:41.881000+00:00,2026-05-27 11:26:41.881000+00:00,NaN,False,False,False
6,6a0eac6b174ad3c431d9e988,DEPOSIT,TXNY_MPNZC5TD_280E488153E1,5000,initiated,2026-05-27 11:27:25.403000+00:00,2026-05-27 11:27:25.403000+00:00,NaN,False,False,False
9,6a0eac6b174ad3c431d9e988,DEPOSIT,TXNY_MPQWTRA2_F094F95275C2,100000,initiated,2026-05-29 12:40:26.224000+00:00,2026-05-29 12:40:26.224000+00:00,NaN,False,False,False
10,6a0eac6b174ad3c431d9e988,DEPOSIT,TXNY_MPQWTRPJ_661944F85CFF,100000,initiated,2026-05-29 12:40:26.597000+00:00,2026-05-29 12:40:26.597000+00:00,NaN,False,False,False
11,6a0eac6b174ad3c431d9e988,DEPOSIT,TXNY_MPQWV783_0082ED8AD471,5000,initiated,2026-05-29 12:41:33.360000+00:00,2026-05-29 12:41:33.360000+00:00,NaN,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...
1152,6a2f98f60731c9a7d6634114,DEPOSIT,TXNY_MQKKJ5MN_BE2C879EB4C7,100,completed,2026-06-19 06:49:21.383000+00:00,2026-06-19 06:52:22.914000+00:00,NaN,False,True,False
1153,6a2f98f60731c9a7d6634114,DEPOSIT,TXNY_MQKKK828_D1AC4D51389C,100,completed,2026-06-19 06:50:11.228000+00:00,2026-06-19 06:53:11.846000+00:00,NaN,False,True,False
1154,6a2f98f60731c9a7d6634114,DEPOSIT,TXNY_MQKKMZZ6_4E8183BECA5C,100,completed,2026-06-19 06:52:20.700000+00:00,2026-06-19 06:59:11.367000+00:00,NaN,False,True,False
1155,6a2f98f60731c9a7d6634114,WITHDRAWAL,TXNY_MQKKVP7I_5FFF733A1C06,2100,completed,2026-06-19 06:59:05.765000+00:00,2026-06-19 06:59:06.864000+00:00,756777777,False,False,True


,player_key,ticket_id,stake,created_at,status
22,6a0eac6b174ad3c431d9e988,10cd3517-823a-4de5-8b65-b688baf78cca,1000,2026-05-31 10:26:01.500000+00:00,SETTLED
62,6a0eac6b174ad3c431d9e988,f13b6bb5-a089-4cbc-b817-96a5ad5c52c9,1000,2026-06-01 10:49:50.283000+00:00,SETTLED
63,6a0eac6b174ad3c431d9e988,7da62773-0ed4-4cd3-be00-685d800941fd,1000,2026-06-01 10:50:47.118000+00:00,SETTLED
201,6a0eac6b174ad3c431d9e988,327e7db9-406a-43d2-9945-775dcdf87f8d,1000,2026-06-05 07:02:38.495000+00:00,SETTLED
202,6a0eac6b174ad3c431d9e988,fa1b2493-8ccf-4283-a691-73a3cf52bdab,1000,2026-06-05 07:02:58.397000+00:00,SETTLED
...,...,...,...,...,...
395,6a2f98f60731c9a7d6634114,24df50f8-5884-46f1-affa-98d389c174c9,1000,2026-06-15 08:50:05.507000+00:00,SETTLED
396,6a2f98f60731c9a7d6634114,a3b33332-72f5-46b4-bce6-1fdb160a9352,2322,2026-06-15 08:50:33.378000+00:00,SETTLED
434,6a2f98f60731c9a7d6634114,9c9638f9-88e5-4e6a-ac03-85a8065c603a,1000,2026-06-19 07:17:59.151000+00:00,ACCEPTED
435,6a2f98f60731c9a7d6634114,4df15db6-0e8d-4410-9b26-e57fec6a442a,1000,2026-06-19 07:18:08.394000+00:00,ACCEPTED


## FINDINGS ? Payment features ? frozen Phase 3 snapshot ? 2026-06-22
- Verified: with `ma_`, `pay_`, and `bet_` now all present, the combined output has 116 player rows and continues to carry the reviewed 14 `pay_` features with the same frozen payment counts.
- Verified: `pay_deposit_then_exit_flag` is non-null for 8/116 and non-zero for 1; `pay_intervening_turnover_ratio` is non-null for 1/116 and non-zero for 0. The flagship is largely dormant by design.
- Verified: 96/116 players receive `casino_activity_not_observable` on both turnover-dependent features because they have no sportsbook history and player-level casino play is not observable. This is an honest null, not a confident false/zero.
- Verified: `pay_min_minutes_deposit_to_withdrawal` is non-null/non-zero for 10/116.
- Verified: `pay_third_party_withdrawal_flag` and `pay_third_party_withdrawal_count` are measurable for 10/116 and non-zero for 0; every completed withdrawal in this snapshot has `is_third_party_recipient=False`.
- Verified: `pay_withdrawal_to_deposit_ratio` is non-null for 26/116 and non-zero for 9 after the 10,000 UGX placeholder denominator gate.
- Verified: `pay_fast_withdrawal_count` is non-null for 10/116 and non-zero for 6.
- Verified: `pay_manual_reconciliation_count` and `pay_manual_reconciliation_ratio` are non-null for 38/116 and non-zero for 5.
- Verified: `pay_declined_withdrawal_count` is non-null for 11/116 and non-zero for 3; `pay_failed_withdrawal_count` is non-null for 11/116 and non-zero for 5.
- Verified: `pay_net_money_flow` is non-null for 116/116 and non-zero for 38; both distinct-instrument features are non-null for 116/116 and non-zero for 39.
- Verified: `ma_` and `pay_` receive the same `WithdrawalContext`; completed-withdrawal filtering and recipient linkage are computed once by the combined builder.
- Awkwardness: `ma_withdrawal_recipient_shared_count` measures cross-player sharing while `pay_third_party_withdrawal_*` measures recipient-vs-own-phone. They share the authoritative recipient population/index, but their scalar meanings are intentionally different.
- Gaps: player-level casino telemetry is absent, so the future confirmed-zero-play branch remains unreachable. Deposit structuring and bank features remain TODOs only. All payment thresholds are uncalibrated PLACEHOLDER values for production calibration.


## Betting Feature Coverage and Evidence Integrity

In [8]:
bet_coverage_rows = []
for feature_name, spec in BET_FEATURE_SPECS.items():
    values = player_features[feature_name]
    non_zero = (
        int(values.dropna().astype(bool).sum())
        if spec.value_kind == "bool"
        else int(values.dropna().ne(0).sum())
    )
    bet_coverage_rows.append(
        {
            "feature_name": feature_name,
            "strength": spec.strength,
            "scoring_role": spec.scoring_role,
            "players": len(values),
            "non_null": int(values.notna().sum()),
            "non_zero": non_zero,
            "null": int(values.isna().sum()),
            "null_reasons": player_features[f"{feature_name}__null_reason"].value_counts(dropna=True).to_dict(),
        }
    )
bet_coverage = pd.DataFrame(bet_coverage_rows)
display(bet_coverage)

sportsbook_active = sportsbook_active_players(bets_raw, set(player_features.player_key.astype(str)))
bet_concentration_active = set(
    player_features.loc[player_features["bet_game_type_concentration"].notna(), "player_key"].astype(str)
)
pay_turnover_observable = set(
    player_features.loc[
        player_features["pay_deposit_then_exit_flag__null_reason"]
        .fillna("measured")
        .ne("casino_activity_not_observable"),
        "player_key",
    ].astype(str)
)
print("sportsbook_active_players", len(sportsbook_active))
print("bet_game_type_concentration active", len(bet_concentration_active))
print("pay turnover-observable players", len(pay_turnover_observable))
assert sportsbook_active == bet_concentration_active == pay_turnover_observable
print("VERIFIED: bet_game_type_concentration and pay_ casino-null gate use the same sportsbook-active set")

for row in player_features.itertuples(index=False):
    values = row._asdict()
    for feature_name, spec in BET_FEATURE_SPECS.items():
        value = values[feature_name]
        if spec.scoring_role == "scoring" and pd.notna(value) and bool(value):
            assert evidence_lookup[(values["player_key"], feature_name)]
print("VERIFIED: every non-zero scoring bet_ feature has non-empty evidence")


,feature_name,strength,scoring_role,players,non_null,non_zero,null,null_reasons
0,bet_win_rate_vs_volume,strong,scoring,116,2,2,114,"{'no_bets': 96, 'insufficient_settled_bets': 18}"
1,bet_timing_regularity,strong,scoring,116,3,3,113,"{'no_bets': 96, 'insufficient_bets_for_timing': 17}"
2,bet_stake_volatility,moderate,supporting,116,9,6,107,"{'no_bets': 96, 'insufficient_bets_for_volatility': 11}"
3,bet_bonus_funded_stake_share,moderate,supporting,116,20,7,96,{'no_bets': 96}
4,bet_game_type_concentration,moderate,context_only,116,20,20,96,{'no_bets': 96}
5,bet_avg_odds,weak,context_only,116,20,20,96,{'no_bets': 96}
6,bet_odds_profile,weak,context_only,116,20,18,96,{'no_bets': 96}
7,bet_count,weak,context_only,116,116,20,0,{}
8,bet_active_days,weak,context_only,116,116,20,0,{}
9,bet_void_rate,weak,context_only,116,20,3,96,{'no_bets': 96}


sportsbook_active_players 20
bet_game_type_concentration active 20
pay turnover-observable players 20
VERIFIED: bet_game_type_concentration and pay_ casino-null gate use the same sportsbook-active set
VERIFIED: every non-zero scoring bet_ feature has non-empty evidence


## Betting Hand Reconciliation Against Frozen Parquet

In [9]:
bet_signal_players = list(
    dict.fromkeys(
        player_features.loc[
            player_features["bet_win_rate_vs_volume"].notna()
            | player_features["bet_timing_regularity"].notna()
            | player_features["bet_bonus_funded_stake_share"].fillna(0).gt(0)
            | player_features["bet_void_rate"].fillna(0).gt(0),
            "player_key",
        ].astype(str).tolist()
    )
)[:3]
print("bet_signal_players", bet_signal_players)
display(
    player_features.loc[
        player_features.player_key.isin(bet_signal_players),
        ["player_key"] + list(BET_FEATURE_SPECS),
    ]
)

bet_focus_evidence = feature_evidence[
    feature_evidence.player_key.isin(bet_signal_players)
    & feature_evidence.feature_name.str.startswith("bet_")
].copy()
bet_focus_evidence["parsed_evidence"] = bet_focus_evidence.feature_evidence.map(json.loads)
display(
    bet_focus_evidence.loc[
        bet_focus_evidence.parsed_evidence.map(bool),
        ["player_key", "feature_name", "parsed_evidence"],
    ]
)

display(
    bets_raw.loc[
        bets_raw.player_key.isin(bet_signal_players),
        [
            "player_key", "ticket_id", "game_type", "status", "result",
            "stake", "stake_bonus", "is_free_bet", "total_odds", "created_at",
        ],
    ].sort_values(["player_key", "created_at", "ticket_id"])
)


bet_signal_players ['6a0ea9ff174ad3c431d9e16d', '6a215a2a7364bea0f4f5a90f', '6a21639b7364bea0f4f5b356']


,player_key,bet_win_rate_vs_volume,bet_timing_regularity,bet_stake_volatility,bet_bonus_funded_stake_share,bet_game_type_concentration,bet_avg_odds,bet_odds_profile,bet_count,bet_active_days,bet_void_rate
0,6a0ea9ff174ad3c431d9e16d,0.333333,1.198042,0.349556,0.0,1.0,3.682903,2.653999,31,1,0.032258
27,6a215a2a7364bea0f4f5a90f,<NA>,<NA>,<NA>,0.0,1.0,6.07,4.040748,9,3,0.111111
30,6a21639b7364bea0f4f5b356,0.418502,6.788538,0.06481,0.008326,1.0,3.995397,3.457931,239,10,0.033473


,player_key,feature_name,parsed_evidence
0,6a0ea9ff174ad3c431d9e16d,bet_active_days,"[{'active_dates': ['2026-05-31'], 'ticket_ids': ['0150ed34-69af-4cb4-814c-88339c4071ad', '05d2cf0e-af72-4f86-b086-b9795b2cbc9a', '097d6717-b3da-4065-bd09-4b..."
1,6a0ea9ff174ad3c431d9e16d,bet_avg_odds,"[{'odds_count': 31, 'ticket_ids': ['0150ed34-69af-4cb4-814c-88339c4071ad', '05d2cf0e-af72-4f86-b086-b9795b2cbc9a', '097d6717-b3da-4065-bd09-4b4214b87107', '..."
3,6a0ea9ff174ad3c431d9e16d,bet_count,"[{'ticket_ids': ['0150ed34-69af-4cb4-814c-88339c4071ad', '05d2cf0e-af72-4f86-b086-b9795b2cbc9a', '097d6717-b3da-4065-bd09-4b4214b87107', '0c0b053f-7d25-4b3a..."
4,6a0ea9ff174ad3c431d9e16d,bet_game_type_concentration,"[{'dominant_game_type': 'Sports-book', 'dominant_share': 1.0, 'per_game_type': [{'bet_count': 31, 'game_type': 'Sports-book', 'stake_share': 1.0, 'stake_sum..."
5,6a0ea9ff174ad3c431d9e16d,bet_odds_profile,"[{'odds_max': 11.0, 'odds_mean': 3.6829032258064514, 'odds_min': 1.07, 'odds_std': 2.6539990441065773, 'ticket_ids': ['0150ed34-69af-4cb4-814c-88339c4071ad'..."
6,6a0ea9ff174ad3c431d9e16d,bet_stake_volatility,"[{'extreme_ticket_ids': ['57f5a5fd-a781-4318-ba57-de7d12f0fb0d', 'c4cd76b3-ad7d-46b7-9078-ff54d5f79b16'], 'game_type': 'Sports-book', 'max_stake': 3100.0, '..."
7,6a0ea9ff174ad3c431d9e16d,bet_timing_regularity,"[{'coefficient_of_variation': 1.1980416456055618, 'game_type': 'Sports-book', 'gap_seconds': [261.676, 273.203, 153.715, 106.942, 51.287, 89.523, 12.086, 22..."
8,6a0ea9ff174ad3c431d9e16d,bet_void_rate,[{'void_ticket_ids': ['05d2cf0e-af72-4f86-b086-b9795b2cbc9a']}]
9,6a0ea9ff174ad3c431d9e16d,bet_win_rate_vs_volume,"[{'game_type': 'Sports-book', 'reason_text': '10 wins / 30 settled bets', 'settled_bets': 30, 'win_rate': 0.3333333333333333, 'winning_ticket_ids': ['0c0b05..."
891,6a215a2a7364bea0f4f5a90f,bet_active_days,"[{'active_dates': ['2026-05-26', '2026-05-27', '2026-05-29'], 'ticket_ids': ['125d320f-7d23-4cdb-8af4-00363b1dfad6', '48faa45e-c3a6-4d41-825c-994b8c4d61f2',..."


,player_key,ticket_id,game_type,status,result,stake,stake_bonus,is_free_bet,total_odds,created_at
20,6a0ea9ff174ad3c431d9e16d,a3da6651-2ff3-4117-a99b-35f200c1db08,Sports-book,SETTLED,LOSE,1100,0,False,9.00,2026-05-31 10:25:08.766000+00:00
23,6a0ea9ff174ad3c431d9e16d,30e0e442-8dcd-483a-a8ea-1dffb1c97093,Sports-book,SETTLED,WIN,1100,0,False,1.20,2026-05-31 10:29:30.442000+00:00
24,6a0ea9ff174ad3c431d9e16d,2769098b-18b5-42d6-9e3a-32e26cca7709,Sports-book,SETTLED,LOSE,1100,0,False,5.50,2026-05-31 10:34:03.645000+00:00
25,6a0ea9ff174ad3c431d9e16d,e83d1abe-6ebe-4052-8663-b57d23f236c1,Sports-book,SETTLED,LOSE,1100,0,False,3.40,2026-05-31 10:36:37.360000+00:00
26,6a0ea9ff174ad3c431d9e16d,68cb88c2-1492-4d57-a65d-68715d53c2f2,Sports-book,SETTLED,WIN,1100,0,False,2.62,2026-05-31 10:38:24.302000+00:00
...,...,...,...,...,...,...,...,...,...,...
424,6a21639b7364bea0f4f5b356,f511dc59-6da9-4819-b144-3ae06cd9e8bf,Sports-book,SETTLED,LOSE,1000,0,False,2.75,2026-06-18 08:44:07.945000+00:00
425,6a21639b7364bea0f4f5b356,b2344352-3502-4369-9e2c-af0fd9f47508,Sports-book,SETTLED,LOSE,1000,0,False,3.40,2026-06-18 08:45:04.130000+00:00
426,6a21639b7364bea0f4f5b356,8be3e8d2-5e81-42a5-9c3b-704bd7c2528e,Sports-book,SETTLED,WIN,1000,0,False,1.83,2026-06-18 08:45:08.954000+00:00
438,6a21639b7364bea0f4f5b356,c4e41f36-aeb6-426a-a1f8-135951249b05,Sports-book,ACCEPTED,NaN,1000,0,False,2.40,2026-06-19 10:25:32.646000+00:00


## FINDINGS ? Betting features ? frozen Phase 3 snapshot ? 2026-06-22
- Verified: all three Phase 3 groups now build together: 116 player rows, 33 reviewed features (9 `ma_` + 14 `pay_` + 10 `bet_`), and 3,828 evidence rows.
- Verified: `bet_win_rate_vs_volume` is non-null for 2/116 and non-zero for 2; 96 players have `no_bets` and 18 have `insufficient_settled_bets`. This dormancy is expected on dev and not a defect.
- Verified: `bet_timing_regularity` is non-null/non-zero for 3/116; 96 players have `no_bets` and 17 have `insufficient_bets_for_timing`. The low-volume gate is doing its job.
- Verified: `bet_stake_volatility` is non-null for 9/116 and non-zero for 6; 96 players have `no_bets` and 11 have `insufficient_bets_for_volatility`.
- Verified: `bet_bonus_funded_stake_share` is non-null for the 20 sportsbook-active players and non-zero for 7; zero is a measured value for players with bets but no bonus-funded stake.
- Verified: `bet_game_type_concentration`, `bet_avg_odds`, and `bet_void_rate` are measured only for the 20 sportsbook-active players. `bet_game_type_concentration` is non-zero for all 20 and is `context_only` in v1.
- Verified: `bet_odds_profile` is non-null for 20/116 and non-zero for 18; `bet_count` and `bet_active_days` are non-null for 116/116 with real zeros for no-bet players.
- Verified: the sportsbook-active set from `bet_game_type_concentration` matches the shared `sportsbook_active_players` helper and the `pay_` turnover-observable population: 20/116 players.
- Awkwardness: wide feature output requires one scalar per feature, while the dictionary asks for per-`game_type` computation. The implementation keeps scalar values as the dominant/worst measured per-game statistic and stores the full per-game breakdown in evidence.
- Gaps: all casino game-exploitation features remain TODO-only until per-round logging exists. All betting thresholds are uncalibrated PLACEHOLDER values for production calibration.
- Decisions needed: None for this group build. All three Phase 3 feature groups now exist; next gate is full feature-table assembly review and Phase 4 scoring design.
